# flashattention-cuda — Colab bootstrap (T4)

One pass: confirm the GPU → build the kernel → predict the roofline → test vs SDPA → benchmark.

**Runtime → Change runtime type → T4 GPU** before running. Everything below runs on the GPU;
nothing here works on a CPU-only runtime.

## 0. Confirm the hardware (Step 0 of the brief)
We record GPU model, compute capability, and clocks — every benchmark row must carry these,
and the free-tier T4 thermally throttles.

In [1]:
!nvidia-smi --query-gpu=name,compute_cap,clocks.sm,clocks.max.sm,memory.total,temperature.gpu --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| capability', torch.cuda.get_device_capability())

name, compute_cap, clocks.current.sm [MHz], clocks.max.sm [MHz], memory.total [MiB], temperature.gpu
Tesla T4, 7.5, 300 MHz, 1590 MHz, 15360 MiB, 40
torch 2.11.0+cu128 | cuda 12.8 | capability (7, 5)


## 1. Get the repo
`REPO_URL` is already set to the public repo, so the clone just works. (If you fork it, point
`REPO_URL` at your fork.) The repo root is added to `sys.path` so the notebook can import it.

In [2]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works on Colab
import os, sys, subprocess
if not os.path.isdir('flashattention-cuda'):
    subprocess.run(['git', 'clone', REPO_URL], check=True)
os.chdir('flashattention-cuda')
sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

cwd /content/flashattention-cuda


## 2. Predict the roofline BEFORE running anything
The per-step loop starts here: predict the limiter, then check it against reality below.

In [3]:
!python -m roofline.predict --arch sm_75 --shape 1x8x2048x64 --precision fp32 --materialize-s
print()
!python -m roofline.predict --arch sm_75 --shape 1x8x2048x128 --precision fp32 --materialize-s

arch        : Tesla T4 (sm_75)
shape       : B=1 H=8 N=2048 d=64  precision=fp32  materialize_S=True  tile=1x1
LIMITER     : HBM   (predicted lower bound 109.065 ms)
  t_mma     :    1.060 ms   util   1.0%
  t_hbm     :  109.065 ms   util 100.0%
  t_mufu    :    0.033 ms   util   0.0%
intensity   : 0.2 FLOP/byte   (arch ridge 25.3; BELOW -> memory-bound)

arch        : Tesla T4 (sm_75)
shape       : B=1 H=8 N=2048 d=128  precision=fp32  materialize_S=True  tile=1x1
LIMITER     : HBM   (predicted lower bound 216.452 ms)
  t_mma     :    2.121 ms   util   1.0%
  t_hbm     :  216.452 ms   util 100.0%
  t_mufu    :    0.033 ms   util   0.0%
intensity   : 0.2 FLOP/byte   (arch ridge 25.3; BELOW -> memory-bound)


## 3. Build the v1 kernel (JIT)
First call compiles with nvcc (~1 min); cached afterwards. `verbose=True` prints the build.

In [4]:
# Clean build: a failed compile (e.g. ninja missing on the first try) leaves a stale cache dir
# with a version stamp but no .so, so torch SKIPS the rebuild and then fails to import. Remove
# any partial fa_* build dir (one with no compiled .so); a good cached build is kept, so re-runs
# stay fast.
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_*')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True)
        print('cleaned stale build:', d)
print('clean-build check done')

clean-build check done


In [5]:
# cpp_extension.load() compiles via ninja, which Colab doesn't always ship — install it first.
!pip install -q ninja
from bindings.load import build_kernel
mod = build_kernel('v1_naive')
print('built:', mod)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 6.7 MB/s eta 0:00:00
built: <module 'fa_v1_naive' from '/root/.cache/torch_extensions/py312_cu128/fa_v1_naive/fa_v1_naive.so'>


## 4. Correctness vs SDPA (documented tolerance: atol/rtol 1e-4)

In [6]:
!python -m pytest tests/ -q

..........................                                               [100%]
26 passed in 90.46s (0:01:30)


## 5. Benchmark vs SDPA across the sweep
Expect to be **slower than SDPA** — SDPA is already a fused efficient kernel. This is the
'before'. Paste the numbers into `docs/results.md` and compare to the roofline prediction.

In [7]:
!python -m bench.harness --backend v1_naive --precision fp32

# device: Tesla T4 (sm_75)  clock~300/1590MHz  backend=v1_naive  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
ninja: no work to do.
        1x8x512x64 |   6.465/ 14.588 |   0.198/  4.068 |    0.03x |    6.336e+05 | HBM (~6.82ms)
       1x8x512x128 |  10.138/ 10.365 |   0.338/  5.402 |    0.03x |    4.040e+05 | HBM (~13.53ms)
       1x8x2048x64 |  87.152/ 90.636 |   3.106/  3.397 |    0.04x |    1.880e+05 | HBM (~109.07ms)
      1x8x2048x128 | 169.020/179.415 |   5.769/  6.492 |    0.03x |    9.694e+04 | HBM (~216.45ms)
       1x8x8192x64 | 1633.956/1797.442 |  52.478/ 52.867 |    0.03x |    4.011e+04 | HBM (~1744.88ms)
      1x8x8192x128 | 3090.600/3099.048 | 102.601/104.512 |    0.03x |    2.120e+04 | HBM (~3462.92ms)


## 6. (Optional) Nsight Compute capture
Confirms the limiter: DRAM throughput near peak at d=64 (the bandwidth wall), low MMA/MUFU.
`ncu` may need a GPU runtime that allows profiling; see profiling/GUIDE.md.

In [8]:
!bash profiling/capture.sh v1_naive || echo 'ncu unavailable on this runtime; read GUIDE.md'

==PROF== Connected to process 7845 (/usr/bin/python3.12)
# device: Tesla T4 (sm_75)  clock~300/1590MHz  backend=v1_naive  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
ninja: no work to do.
==PROF== Profiling "<unnamed>::qk_kernel(const float *, const float *, float *, int, int, int, float, long)": 0%....50%....100% - 31 passes
==PROF== Profiling "<unnamed>::softmax_kernel(float *, int, int, bool, long)": 0%....50%....100% - 31 passes
==PROF== Profiling "<unnamed>::pv_kernel(const float *, const float *, float *, int, int, int, long)": 0%....50%....100% - 31 passes
        1x8x512x64 |   5.933/ 10.501 |   0.320/  1.033 |    0.05x |    6.904e+05 | HBM (~6.82ms)
       1x8x512x128 |  11.013/ 11.246 |   0.475/  0.526 |    0.04x |    3.719e+05 | HBM (~13.53ms)
       1x8x2048x64 |  94.596/ 98.021 |   3.598/  3.812 |    0.04x |    1.732e+05 | HBM (~109.07ms)
      1x8x2048x128 | 183.980/190.527 |   6.722/  7.18

In [9]:
!git pull origin main

From https://github.com/gkienpham-cmd/flashattention-cuda
 * branch            main       -> FETCH_HEAD
Already up to date.


In [10]:
!python -m pytest -q

..........................                                               [100%]
26 passed in 2.32s


In [11]:
!python -m bench.harness --backend v2_tiled --precision fp32

# device: Tesla T4 (sm_75)  clock~345/1590MHz  backend=v2_tiled  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
ninja: no work to do.
        1x8x512x64 |   3.674/  3.713 |   0.338/  0.382 |    0.09x |    1.115e+06 | HBM (~0.21ms)
       1x8x512x128 |   3.764/  3.830 |   0.383/  0.539 |    0.10x |    1.088e+06 | HBM (~0.53ms)
       1x8x2048x64 |  34.649/ 35.857 |   3.437/  3.547 |    0.10x |    4.729e+05 | HBM (~3.37ms)
      1x8x2048x128 |  60.560/ 61.202 |   6.509/  7.078 |    0.11x |    2.705e+05 | HBM (~8.41ms)
       1x8x8192x64 | 654.634/887.868 |  52.987/ 53.326 |    0.08x |    1.001e+05 | HBM (~53.74ms)
      1x8x8192x128 | 1057.405/1065.722 | 104.871/105.739 |    0.10x |    6.198e+04 | HBM (~134.32ms)


In [12]:
!python -m bench.harness --backend v1_naive --precision fp32

# device: Tesla T4 (sm_75)  clock~525/1590MHz  backend=v1_naive  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
ninja: no work to do.
        1x8x512x64 |   5.951/ 10.411 |   0.207/  0.233 |    0.03x |    6.883e+05 | HBM (~6.82ms)
       1x8x512x128 |  11.240/ 11.491 |   0.363/  0.370 |    0.03x |    3.644e+05 | HBM (~13.53ms)
       1x8x2048x64 |  98.880/101.165 |   3.584/  3.917 |    0.04x |    1.657e+05 | HBM (~109.07ms)
      1x8x2048x128 | 185.567/190.204 |   6.574/  7.262 |    0.04x |    8.829e+04 | HBM (~216.45ms)
       1x8x8192x64 | 1627.252/1646.571 |  52.860/ 53.200 |    0.03x |    4.027e+04 | HBM (~1744.88ms)
      1x8x8192x128 | 3120.577/3145.688 | 104.577/107.950 |    0.03x |    2.100e+04 | HBM (~3462.92ms)


In [13]:
!bash profiling/capture.sh v2_tiled || echo 'ncu unavailable on this runtime'

==PROF== Connected to process 11208 (/usr/bin/python3.12)
# device: Tesla T4 (sm_75)  clock~345/1590MHz  backend=v2_tiled  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
ninja: no work to do.
==PROF== Profiling "void <unnamed>::qk_tiled_kernel<(int)64, (int)64, (int)64>(const float *, const float *, float *, int, int, float)": 0%....50%....100% - 31 passes
==PROF== Profiling "<unnamed>::softmax_kernel(float *, int, int, bool, long)": 0%....50%....100% - 31 passes
==PROF== Profiling "void <unnamed>::pv_tiled_kernel<(int)64, (int)64, (int)64>(const float *, const float *, float *, int, int)": 0%....50%....100% - 31 passes
        1x8x512x64 |   4.358/  4.384 |   0.515/  0.916 |    0.12x |    9.398e+05 | HBM (~0.21ms)
       1x8x512x128 |   3.779/  3.844 |   0.502/  0.558 |    0.13x |    1.084e+06 | HBM (~0.53ms)
       1x8x2048x64 |  34.207/ 35.215 |   3.514/  3.627 |    0.10x |    4.790e+05 | HBM (~3.37ms)
 

In [14]:
!git pull origin main
!pip install -q ninja
!which ncu && ncu --version

From https://github.com/gkienpham-cmd/flashattention-cuda
 * branch            main       -> FETCH_HEAD
Already up to date.
/usr/local/cuda/bin/ncu
NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.1.1.0 (build 35528883) (public-release)


In [15]:
!bash profiling/capture.sh v1_naive

==ERROR== File v1_naive.ncu-rep already exists. Use '-f' for overwriting the file.


In [16]:
!bash profiling/capture.sh v2_tiled

==ERROR== File v2_tiled.ncu-rep already exists. Use '-f' for overwriting the file.


In [17]:
!ls -la profiling/raw/
!git add -f profiling/raw/v1_naive.ncu-rep profiling/raw/v2_tiled.ncu-rep
!git -c user.email="pgkien11@gmail.com" -c user.name="Kien Pham" commit -m "Step 2 profiling: v1/v2 ncu captures"
!git push origin main

total 5092
drwxr-xr-x 2 root root    4096 Jun 18 19:00 .
drwxr-xr-x 3 root root    4096 Jun 18 18:47 ..
-rw-r--r-- 1 root root 2547668 Jun 18 18:47 v1_naive.ncu-rep
-rw-r--r-- 1 root root 2656817 Jun 18 19:00 v2_tiled.ncu-rep
[main ee7dfdc] Step 2 profiling: v1/v2 ncu captures
 2 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 profiling/raw/v1_naive.ncu-rep
 create mode 100644 profiling/raw/v2_tiled.ncu-rep
fatal: could not read Username for 'https://github.com': No such device or address


In [18]:
!ncu -i profiling/raw/v1_naive.ncu-rep --page raw --csv \
  --metrics dram__bytes_read.sum,gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed,sm__throughput.avg.pct_of_peak_sustained_elapsed,sm__warps_active.avg.pct_of_peak_sustained_active

"ID","Process ID","Process Name","Host Name","Kernel Name","Context","Stream","Block Size","Grid Size","Device","CC","dram__bytes_read.sum","gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed","sm__throughput.avg.pct_of_peak_sustained_elapsed","sm__warps_active.avg.pct_of_peak_sustained_active"
"","","","","","","","","","","","Mbyte","%","%","%"
"0","7845","python3.12","127.0.0.1","<unnamed>::qk_kernel(const float *, const float *, float *, int, int, int, float, long)","1","7","(256, 1, 1)","(8192, 1, 1)","0","7.5","3.679840","0.342790","6.122648","98.249120"
"1","7845","python3.12","127.0.0.1","<unnamed>::softmax_kernel(float *, int, int, bool, long)","1","7","(256, 1, 1)","(16, 1, 1)","0","7.5","49.827328","14.352969","2.942835","24.763133"
"2","7845","python3.12","127.0.0.1","<unnamed>::pv_kernel(const float *, const float *, float *, int, int, int, long)","1","7","(256, 1, 1)","(1024, 1, 1)","0","7.5","18.364576","7.231296","80.534369","94.811698"


In [19]:
!ncu -i profiling/raw/v2_tiled.ncu-rep --page raw --csv \
  --metrics dram__bytes_read.sum,gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed,sm__throughput.avg.pct_of_peak_sustained_elapsed,sm__warps_active.avg.pct_of_peak_sustained_active

"ID","Process ID","Process Name","Host Name","Kernel Name","Context","Stream","Block Size","Grid Size","Device","CC","dram__bytes_read.sum","gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed","sm__throughput.avg.pct_of_peak_sustained_elapsed","sm__warps_active.avg.pct_of_peak_sustained_active"
"","","","","","","","","","","","Mbyte","%","%","%"
"0","11208","python3.12","127.0.0.1","void <unnamed>::qk_tiled_kernel<64, 64, 64>(const float *, const float *, float *, int, int, float)","1","7","(256, 1, 1)","(8, 8, 8)","0","7.5","3.114304","1.235095","6.611886","47.651796"
"1","11208","python3.12","127.0.0.1","<unnamed>::softmax_kernel(float *, int, int, bool, long)","1","7","(256, 1, 1)","(16, 1, 1)","0","7.5","49.909024","14.756893","2.989013","24.784773"
"2","11208","python3.12","127.0.0.1","void <unnamed>::pv_tiled_kernel<64, 64, 64>(const float *, const float *, float *, int, int)","1","7","(256, 1, 1)","(8, 8, 1)","0","7.5","13.485568","6.237783","63.029795","42.006387"
